In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="hf_pipeline_mrpc_bidirectional_order_average",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = model.config.id2label
positive_label = id2label[1]

try:
    clf = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        top_k=None,
        function_to_apply="softmax",
        device=device,
    )
except Exception:
    clf = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        top_k=None,
        function_to_apply="softmax",
    )

print(model_name)
print(id2label)
print("positive_label:", positive_label)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

textattack/distilbert-base-uncased-MRPC
{0: 'LABEL_0', 1: 'LABEL_1'}
positive_label: LABEL_1


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
orig_inputs = [{"text": s1, "text_pair": s2} for s1, s2 in zip(sent1, sent2)]
swap_inputs = [{"text": s2, "text_pair": s1} for s1, s2 in zip(sent1, sent2)]

batch_size = 64

def get_positive_probs(examples, desc):
    probs = []
    for i in tqdm(range(0, len(examples), batch_size), desc=desc):
        batch = examples[i:i + batch_size]
        outputs = clf(
            batch,
            batch_size=batch_size,
            truncation=True,
            max_length=128,
            padding=True,
        )
        for out in outputs:
            score_map = {item["label"]: float(item["score"]) for item in out}
            probs.append(score_map[positive_label])
    return np.array(probs, dtype=np.float32)

orig_pos_probs = get_positive_probs(orig_inputs, "original_order")
swap_pos_probs = get_positive_probs(swap_inputs, "swapped_order")

avg_pos_probs = (orig_pos_probs + swap_pos_probs) / 2.0
y_pred = (avg_pos_probs >= 0.50).astype(np.int64)

print("done")


original_order:   0%|          | 0/7 [00:00<?, ?it/s]

swapped_order:   0%|          | 0/7 [00:00<?, ?it/s]

done


In [ ]:

vault.create_record_list("distilbert_swap_prediction_values", column_names=["prediction", "pos_probs", "swap_pos_probs"])

for i in range(len(y_pred)):
    vault.append_record("distilbert_swap_prediction_values", 
                        {
                            "prediction": y_pred[i],
                            "pos_probs": float(orig_pos_probs[i]),
                            "swap_pos_probs": float(swap_pos_probs[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT distilbert_swap_prediction_values"
embedding = get_embeddings(description)
vault.create_description("distilbert_swap_prediction_values", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_swap_prediction_values", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.8504901960784313, 'f1': 0.897822445561139}
                precision    recall  f1-score   support

not_paraphrase       0.88      0.61      0.72       129
    paraphrase       0.84      0.96      0.90       279

      accuracy                           0.85       408
     macro avg       0.86      0.79      0.81       408
  weighted avg       0.85      0.85      0.84       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("orig_paraphrase_prob:", float(orig_pos_probs[i]))
    print("swap_paraphrase_prob:", float(swap_pos_probs[i]))
    print("avg_paraphrase_prob:", float(avg_pos_probs[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", id2label[int(y_pred[i])])


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
orig_paraphrase_prob: 0.983514666557312
swap_paraphrase_prob: 0.9827525615692139
avg_paraphrase_prob: 0.9831336140632629
true: 1 pred: 1 label: LABEL_1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
orig_paraphrase_prob: 0.18237046897411346
swap_paraphrase_prob: 0.19850967824459076
avg_paraphrase_prob: 0.1904400736093521
true: 0 pred: 0 label: LABEL_0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss fr

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("orig_paraphrase_prob:", float(orig_pos_probs[i]))
    print("swap_paraphrase_prob:", float(swap_pos_probs[i]))
    print("avg_paraphrase_prob:", float(avg_pos_probs[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


num_errors: 61
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
orig_paraphrase_prob: 0.929533839225769
swap_paraphrase_prob: 0.8762917518615723
avg_paraphrase_prob: 0.9029127955436707
true: 0 pred: 1
idx: 26
sentence1: Cooley said he expects Muhammad will similarly be called as a witness at a pretrial hearing for Malvo .
sentence2: Lee Boyd Malvo will be called as a witness Wednesday in a pretrial hearing for fellow sniper suspect John Allen Muhammad .
orig_paraphrase_prob: 0.8425254225730896
swap_paraphrase_prob: 0.9128895998001099
avg_paraphrase_prob: 0.8777074813842773
true: 0 pred: 1
idx: 35
sentence1: Bush wanted " to see an aircraft landing the same way that the pilots saw an aircraft landing , " White House press secretary Ari Fleischer said yesterday .
sentence2: On Tue

In [8]:

vault.create_record_list("hf_pipeline_mrpc_bidirectional_order_average_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("hf_pipeline_mrpc_bidirectional_order_average_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert_swap_prediction_values": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_bidirectional_order_average_summary"
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_mrpc_bidirectional_order_average_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_mrpc_bidirectional_order_average_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'textattack/distilbert-base-uncased-MRPC',
 'device': 'mps',
 'aggregation': 'mean_bidirectional_pair_scores',
 'num_examples': 408,
 'accuracy': 0.8504901960784313,
 'f1': 0.897822445561139}

In [ ]:
description = "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_bidirectional_order_average process/notebook" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_mrpc_bidirectional_order_average", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_mrpc_bidirectional_order_average", cat, embedding, prop)